# Concept-Aware Training — Research Tasks 9–12 (Plan A)

**The decisive round.** A July 2026 code audit found **three** training-side bugs in the concept
pipeline (all fixed 2026-07-20 — see `PlanA.md` §1.1). Because of them, the revised-round (Tasks 5–8)
loss numbers are **bug-contaminated and do not show the loss objective fails** — it was never fairly
tested. This notebook runs the clean test.

| Bug | Where | Effect | Status |
|---|---|---|---|
| #1 bare-token convention | `*_trainer.py` `_get_single_token_id` | supervised ids the model never emits mid-sentence | **fixed** → `_get_concept_first_token_id` (leading space, deduped) |
| #2 PAD-label contamination | all 4 run scripts `tokenize_function` | ~88% of positions trained to predict `[PAD]`; slot fights CLM term | **fixed** → pad `attention_mask=0`, `labels=-100` |
| #3 zero-gradient objective | `custom_trainer.py`/`hierarchical_trainer.py` (`no_grad`) | `syn_ncp`/`hyp_ncp` were CLM-on-concept-CSV, never concept-trained | **retired** → use `diff_ncp` as corrected NCP |

`eval_concept_ppl_v2.py` was audited line-by-line and is **correct** — the bugs are all training-side.

### Task map

| Task | What | Cost |
|---|---|---|
| **9**  | Clean training arms: A0 (pure vanilla), A1 (data-aug), A2 (fixed set-marginal loss), D2 (α=0 forgetting control) — 3 seeds each | GPU, ~9 runs |
| **10** | Diagnostics: D1 convention+PAD mass probe (mechanism figure), D3 audit sampler, paired per-slot stats | mostly CPU |
| **11** | Intrinsic dual eval (v2) + paired stats + master table | GPU eval |
| **12** | Downstream (SNLI probe + low-res) + representation/retention battery (R1 synonym-invariance, R2 STS-B, R3 ARC/MMLU retention) | GPU |

### Drive-quota rule (fixes the earlier blowup)
Every training call uses `--save_only_model True` (no `optimizer.pt` / `scheduler.pt` / `rng_state.pth`
/ `scaler.pt`) **and** trains to **local** `/content` scratch; only whitelisted model + JSON files are
copied to Drive by `push_model_to_drive()`. State files never touch Drive.

**Before running:** push the fixed scripts to GitHub (Colab pulls from there). Setup cell S4 hard-fails
if the fixes are absent.

---
## 0. Setup

In [1]:
# S1: GPU check
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only')
# CPU-only runtime is fine (and 0 GPU credits) for the CPU cells: D3 sampler, paired-stats,
# A2 data build. Use a GPU runtime (L4 recommended) for training/eval cells.

NVIDIA L4, 23034 MiB, 22564 MiB

CUDA: True | NVIDIA L4


In [2]:
# S2: dependencies. lm-eval is only needed for Task 12 R3 (retention); heavy install, skip if
# you are not running R3 this session.
!pip install -q "transformers>=4.41.0,<5.0.0" datasets accelerate evaluate scipy scikit-learn nltk
# !pip install -q lm-eval   # uncomment for Task 12c (R3 retention benchmarks)
print('deps ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 139.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
deps ready


In [3]:
# S3: mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# S4: clone/pull repo AND verify the three bug-fixes are present (Colab pulls from GitHub —
# push the fixed scripts first or this cell hard-fails).
import os
REPO_URL = 'https://github.com/SharvaGogawale1/concept-aware-training.git'
REPO_DIR = '/content/concept_aware_training'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main
%cd {REPO_DIR}

SCR = f'{REPO_DIR}/transformers/examples/pytorch/language-modeling'
def _has(path, needle):
    return os.path.exists(path) and needle in open(path).read()

checks = {
    'bug#1 fixed (contrastive_trainer)':  _has(f'{SCR}/contrastive_trainer.py', '_get_concept_first_token_id'),
    'bug#1 fixed (differentiable_trainer)': _has(f'{SCR}/differentiable_ncp_trainer.py', '_get_concept_first_token_id'),
    'bug#2 fixed (contrastive run)':      _has(f'{SCR}/run_clm_contrastive.py', 'labels = [-100 if t == pad_id else t for t in ids]'),
    'bug#2 fixed (syn run)':              _has(f'{SCR}/run_clm_syn_custom_loss.py', 'labels = [-100 if t == pad_id else t for t in ids]'),
    'eval v2 present':                    os.path.exists(f'{SCR}/eval_concept_ppl_v2.py'),
}
for k, v in checks.items():
    print(f"  {'OK ' if v else 'MISSING'}  {k}")
assert all(checks.values()), 'Fixes not on the pulled repo — push PlanA trainer fixes to GitHub first.'
print('\nAll Plan-A fixes present.')

Cloning into '/content/concept_aware_training'...
remote: Enumerating objects: 5249, done.
remote: Counting objects: 100% (5249/5249), done.
remote: Compressing objects: 100% (3790/3790), done.
remote: Total 5249 (delta 1508), reused 5133 (delta 1404), pack-reused 0 (from 0)
Receiving objects: 100% (5249/5249), 22.12 MiB | 19.45 MiB/s, done.
Resolving deltas: 100% (1508/1508), done.
/content/concept_aware_training
  OK   bug#1 fixed (contrastive_trainer)
  OK   bug#1 fixed (differentiable_trainer)
  OK   bug#2 fixed (contrastive run)
  OK   bug#2 fixed (syn run)
  OK   eval v2 present

All Plan-A fixes present.


In [5]:
# S5: base model
from huggingface_hub import snapshot_download
MODEL_LOCAL_PATH = '/content/Llama-3.2-1B'
snapshot_download(repo_id='meta-llama/Llama-3.2-1B', local_dir=MODEL_LOCAL_PATH,
                  ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'original/*'])
print('base model ready:', MODEL_LOCAL_PATH)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

LICENSE.txt: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

USE_POLICY.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

base model ready: /content/Llama-3.2-1B


In [6]:
# S6: paths + arms + save helpers
import os, json, shutil, glob
import pandas as pd

REPO_DIR    = '/content/concept_aware_training'
DATA_ROOT   = f'{REPO_DIR}/data'
SCRIPTS_DIR = f'{REPO_DIR}/transformers/examples/pytorch/language-modeling'
OUTPUT_ROOT = '/content/drive/MyDrive/concept_aware_outputs'   # <-- update if needed
CLEAN_OUT   = f'{OUTPUT_ROOT}/clean'            # revised-round checkpoints (A1 seed42 = standard_clm)
PLANA_OUT   = f'{OUTPUT_ROOT}/planA'            # Plan-A checkpoints (model files only)
RESULTS_DIR = f'{OUTPUT_ROOT}/planA_results'    # all JSONs
LOCAL_SCRATCH = '/content/train_scratch'        # ephemeral: training writes here (incl. state files)
MODEL_LOCAL_PATH = '/content/Llama-3.2-1B'
TOKENIZER_PATH   = MODEL_LOCAL_PATH             # ONE canonical tokenizer for every eval
for d in (PLANA_OUT, RESULTS_DIR, LOCAL_SCRATCH):
    os.makedirs(d, exist_ok=True)

# clean-split data (built in the revised round Phase 1)
CLEAN_SYN = f'{DATA_ROOT}/syn/youtube_clean'
CLEAN_HYP = f'{DATA_ROOT}/hyp/youtube_clean'
SYN_CONCEPT_TRAIN = f'{CLEAN_SYN}/context_loss_train.csv'
SYN_CONCEPT_VAL   = f'{CLEAN_SYN}/context_loss_val.csv'
HYP_CONCEPT_VAL   = f'{CLEAN_HYP}/context_loss_val.csv'
SYN_AUG_TRAIN     = f'{CLEAN_SYN}/context_syn_train.txt'   # concept-augmented (A1)
SYN_AUG_VAL       = f'{CLEAN_SYN}/context_syn_val.txt'
SYN_VANILLA_TRAIN = f'{CLEAN_SYN}/vanilla_train.txt'       # pure vanilla, upsampled (A0)
SYN_VANILLA_VAL   = f'{CLEAN_SYN}/vanilla_val.txt'
VANILLA_VAL       = f'{CLEAN_HYP}/vanilla_val.txt'         # NTP eval text (as in revised round)
CONTRA_MERGED_TRAIN = f'{DATA_ROOT}/contrastive/youtube_clean/contrastive_train.csv'
CONTRA_MERGED_VAL   = f'{DATA_ROOT}/contrastive/youtube_clean/contrastive_val.csv'

PRIMARY_SEED = 42          # the core pass trains ONLY this seed (enough for A2-vs-A1)
EXTRA_SEEDS  = [123, 7]     # DEFERRED to the end (error bars / camera-ready)
SEEDS = [PRIMARY_SEED] + EXTRA_SEEDS

# Arms. A1 seed42 reuses the existing standard_clm (it IS the data-aug arm — relabelled here).
ARMS = {'base': MODEL_LOCAL_PATH}
for s in SEEDS:
    ARMS[f'A0_vanilla_s{s}'] = f'{PLANA_OUT}/A0_vanilla_s{s}'
    ARMS[f'A2_fixed_s{s}']   = f'{PLANA_OUT}/A2_fixed_s{s}'
ARMS['A1_aug_s42'] = f'{CLEAN_OUT}/standard_clm'            # reuse
ARMS['A1_aug_s123'] = f'{PLANA_OUT}/A1_aug_s123'
ARMS['A1_aug_s7']   = f'{PLANA_OUT}/A1_aug_s7'
ARMS['D2_alpha0']   = f'{PLANA_OUT}/D2_alpha0'
# corrected-NCP reference from the revised round (gradient-flowing, but carried bugs #1+#2 — kept
# only for orientation; the clean NCP is A2)
ARMS['diff_ncp_old'] = f'{CLEAN_OUT}/diff_ncp'

LABELS = {
    'base': 'Base Llama-3.2-1B (untrained)',
    'A0_vanilla_s42': 'A0 vanilla CLM (s42)', 'A0_vanilla_s123': 'A0 vanilla CLM (s123)', 'A0_vanilla_s7': 'A0 vanilla CLM (s7)',
    'A1_aug_s42': 'A1 data-aug CLM (s42)', 'A1_aug_s123': 'A1 data-aug CLM (s123)', 'A1_aug_s7': 'A1 data-aug CLM (s7)',
    'A2_fixed_s42': 'A2 fixed set-marginal (s42)', 'A2_fixed_s123': 'A2 fixed set-marginal (s123)', 'A2_fixed_s7': 'A2 fixed set-marginal (s7)',
    'D2_alpha0': 'D2 forgetting control (α=0)', 'diff_ncp_old': 'diff_ncp (revised, buggy)',
}

def existing(keys):
    return {k: ARMS[k] for k in keys if os.path.exists(ARMS[k])}

# Files copied to Drive for a trained model — everything EXCEPT optimizer/scheduler/rng/scaler state.
_KEEP = {'config.json','generation_config.json','model.safetensors','model.safetensors.index.json',
         'pytorch_model.bin','tokenizer.json','tokenizer_config.json','special_tokens_map.json',
         'tokenizer.model','added_tokens.json','vocab.json','merges.txt',
         'trainer_state.json','all_results.json','train_results.json','eval_results.json'}

def push_model_to_drive(local_dir, drive_dir):
    os.makedirs(drive_dir, exist_ok=True)
    copied = 0
    for name in os.listdir(local_dir):
        if name in _KEEP:
            shutil.copy2(os.path.join(local_dir, name), os.path.join(drive_dir, name)); copied += 1
    # never copy checkpoint-*/ subfolders or state files
    print(f'  pushed {copied} model/json files -> {drive_dir}')

print('Arms configured. Existing:')
for k, v in ARMS.items():
    print(f"  {'OK' if os.path.exists(v) else '--'}  {k}: {v}")

Arms configured. Existing:
  OK  base: /content/Llama-3.2-1B
  OK  A0_vanilla_s42: /content/drive/MyDrive/concept_aware_outputs/planA/A0_vanilla_s42
  OK  A2_fixed_s42: /content/drive/MyDrive/concept_aware_outputs/planA/A2_fixed_s42
  OK  A0_vanilla_s123: /content/drive/MyDrive/concept_aware_outputs/planA/A0_vanilla_s123
  OK  A2_fixed_s123: /content/drive/MyDrive/concept_aware_outputs/planA/A2_fixed_s123
  OK  A0_vanilla_s7: /content/drive/MyDrive/concept_aware_outputs/planA/A0_vanilla_s7
  OK  A2_fixed_s7: /content/drive/MyDrive/concept_aware_outputs/planA/A2_fixed_s7
  OK  A1_aug_s42: /content/drive/MyDrive/concept_aware_outputs/clean/standard_clm
  OK  A1_aug_s123: /content/drive/MyDrive/concept_aware_outputs/planA/A1_aug_s123
  OK  A1_aug_s7: /content/drive/MyDrive/concept_aware_outputs/planA/A1_aug_s7
  OK  D2_alpha0: /content/drive/MyDrive/concept_aware_outputs/planA/D2_alpha0
  OK  diff_ncp_old: /content/drive/MyDrive/concept_aware_outputs/clean/diff_ncp


In [7]:
# S7: training helper — trains to LOCAL scratch (state files stay ephemeral), copies only model+json
# to Drive. Common flags: save_only_model True (belt), save_total_limit 1, bf16, block_size 128.
def train_arm(arm_key, script, extra_args, seed, train_file, val_file, epochs=3, lr=None):
    local_dir = f'{LOCAL_SCRATCH}/{arm_key}'
    drive_dir = ARMS[arm_key]
    shutil.rmtree(local_dir, ignore_errors=True)
    lr_arg = f' --learning_rate {lr}' if lr else ''
    cmd = (f"python {SCRIPTS_DIR}/{script}"
           f" --model_name_or_path {MODEL_LOCAL_PATH}"
           f" --train_file {train_file} --validation_file {val_file}"
           f" --seed {seed} --num_train_epochs {epochs}{lr_arg}"
           f" --gradient_accumulation_steps 8 --per_device_train_batch_size 2 --per_device_eval_batch_size 2"
           f" --torch_dtype bfloat16 --bf16 True --block_size 128 --auto_find_batch_size True"
           f" --save_total_limit 1 --save_only_model True"   # no optimizer/scheduler/rng/scaler saved
           f" --overwrite_output_dir --do_train --do_eval --report_to none"
           f" {extra_args} --output_dir {local_dir}")
    print(f'\n=== TRAIN {arm_key} (seed {seed}) ===')
    get_ipython().system(cmd)
    if os.path.exists(local_dir):
        push_model_to_drive(local_dir, drive_dir)
    else:
        print('  WARNING: no local output — training may have failed')
print('train_arm() ready')

train_arm() ready


---
## Task 9 — Clean training arms

- **A0** pure-vanilla CLM on `vanilla_train.txt` (upsampled originals). *Caveat:* only ~545 unique
  sentences vs 7,257 augmented — so A1-vs-A0 tests "concept-augmented variants vs repeated originals at
  matched token count", partly a diversity effect. State this in the paper.
- **A1** data-augmentation CLM on `context_syn_train.txt` (this is what `standard_clm` already was —
  seed 42 reused; +2 seeds here).
- **A2** the corrected set-marginal loss: `run_clm_differentiable_ncp.py` (fixed convention + PAD mask)
  with **replay** rows and **LR 1e-5, 2 epochs** to limit forgetting.
- **D2** α=0 control (same A2 pipeline, no concept term) — isolates forgetting from the objective.

In [8]:
# 9a. Build the A2 training CSV = syn concept rows + vanilla replay rows (empty concept set → CLM-only
# for those rows). Columns: text, context_syn. CPU-only.
import pandas as pd, ast, random
df_concept = pd.read_csv(SYN_CONCEPT_TRAIN)
concept_col = 'context_syn' if 'context_syn' in df_concept.columns else df_concept.columns[-1]
df_concept = df_concept.rename(columns={concept_col: 'context_syn'})[['text', 'context_syn']]

vanilla_lines = [l.strip() for l in open(SYN_VANILLA_TRAIN) if l.strip()]
random.seed(42)
# ~1:1 replay, deduped so we don't just re-add the 545 originals many times
replay = list(dict.fromkeys(vanilla_lines))
replay_df = pd.DataFrame({'text': replay, 'context_syn': ['[]'] * len(replay)})

A2_TRAIN = f'{DATA_ROOT}/syn/youtube_clean/A2_train_with_replay.csv'
A2_VAL   = SYN_CONCEPT_VAL
pd.concat([df_concept, replay_df], ignore_index=True).to_csv(A2_TRAIN, index=False)
print(f'A2 train: {len(df_concept)} concept + {len(replay_df)} replay = {len(df_concept)+len(replay_df)} rows -> {A2_TRAIN}')

A2 train: 1795 concept + 545 replay = 2340 rows -> /content/concept_aware_training/data/syn/youtube_clean/A2_train_with_replay.csv


In [ ]:
# 9b. CORE training — seed 42 only (3 runs: A0, A2, D2). A1 seed 42 already exists as standard_clm,
# so it is reused, not retrained. This is all you need for the Week-2 A2-vs-A1 decision.
train_arm('A0_vanilla_s42', 'run_clm.py', extra_args='', seed=PRIMARY_SEED,
          train_file=SYN_VANILLA_TRAIN, val_file=SYN_VANILLA_VAL, epochs=3)
train_arm('A2_fixed_s42', 'run_clm_differentiable_ncp.py', extra_args='--ncp_alpha 0.5',
          seed=PRIMARY_SEED, train_file=A2_TRAIN, val_file=A2_VAL, epochs=2, lr=1e-5)
train_arm('D2_alpha0', 'run_clm_differentiable_ncp.py', extra_args='--ncp_alpha 0.0',
          seed=PRIMARY_SEED, train_file=A2_TRAIN, val_file=A2_VAL, epochs=2, lr=1e-5)
print('\nCore arms trained. A1 seed42 = existing standard_clm (reused).')


=== TRAIN A0_vanilla_s42 (seed 42) ===
2026-07-27 21:30:27.181942: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-27 21:30:27.253878: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO:__main__:Training/evaluation parameters TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0

In [ ]:
# 9c. DEFERRED — extra seeds (123, 7) for error bars. Do NOT run on the first pass; flip the flag
# at camera-ready. ~6 runs (A0x2, A1x2, A2x2). Everything downstream uses existing() so the tables
# stay correct whether 1 or 3 seeds exist.
RUN_EXTRA_SEEDS = True
if RUN_EXTRA_SEEDS:
    for s in EXTRA_SEEDS:
        train_arm(f'A0_vanilla_s{s}', 'run_clm.py', extra_args='', seed=s,
                  train_file=SYN_VANILLA_TRAIN, val_file=SYN_VANILLA_VAL, epochs=3)
        train_arm(f'A1_aug_s{s}', 'run_clm.py', extra_args='', seed=s,
                  train_file=SYN_AUG_TRAIN, val_file=SYN_AUG_VAL, epochs=3)
        train_arm(f'A2_fixed_s{s}', 'run_clm_differentiable_ncp.py', extra_args='--ncp_alpha 0.5',
                  seed=s, train_file=A2_TRAIN, val_file=A2_VAL, epochs=2, lr=1e-5)
else:
    print('Extra seeds deferred (RUN_EXTRA_SEEDS=False). Flip to True at camera-ready for error bars.')


=== TRAIN A0_vanilla_s123 (seed 123) ===
2026-07-29 16:49:12.561726: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-29 16:49:12.632520: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO:__main__:Training/evaluation parameters TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2

---
## Task 10 — Diagnostics

- **D1** mass probe: at each val slot, the model's probability on `[PAD]`, on **bare** first-token ids,
  and on **leading-space** first-token ids of the concept set. This is the mechanism figure — it shows
  the buggy models dumping mass on `[PAD]`/bare ids while clean models put it on the spaced form.
- **D3** audit sampler: writes 50 syn + 50 hyp slots to a CSV for manual "is this positive valid in
  context?" annotation (gates the whole metric).
- **paired stats**: per-slot ΔNLL between arms (paired bootstrap) — the July unpaired CIs spanned two
  orders of magnitude and could resolve nothing.

In [ ]:
# 10a. D1 — convention + PAD mass probe (GPU; ~5-10 min/model). Self-contained.
import torch, ast, json
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

def _first_ids(tok, words, spaced):
    out = []
    for w in words:
        w = str(w).strip()
        if not w:
            continue
        enc = tok((' ' + w) if spaced else w, add_special_tokens=False)['input_ids']
        if enc:
            out.append(enc[0])
    return list(dict.fromkeys(out))

@torch.no_grad()
def d1_probe(ckpt, concept_csv, tok, n_rows=200):
    model = AutoModelForCausalLM.from_pretrained(ckpt, torch_dtype=torch.bfloat16).cuda().eval()
    pad_id = tok.convert_tokens_to_ids('[PAD]')
    df = pd.read_csv(concept_csv)
    col = 'context_syn' if 'context_syn' in df.columns else ('positives' if 'positives' in df.columns else df.columns[-1])
    mp, mb, ms, n = 0.0, 0.0, 0.0, 0
    for _, row in df.head(n_rows).iterrows():
        text = str(row['text']).rstrip()
        if not text:
            continue
        try:
            words = ast.literal_eval(str(row[col]))
        except Exception:
            continue
        if not words:
            continue
        ids = tok(text, add_special_tokens=True, return_tensors='pt')['input_ids'].cuda()
        probs = F.softmax(model(ids).logits[0, -1].float(), dim=-1)
        bare = _first_ids(tok, words, spaced=False)
        spaced = _first_ids(tok, words, spaced=True)
        # Guard: clean models (base/A0/A1) have vocab 128256 and NO [PAD] row, so pad_id is out of
        # range — their pad mass is 0 by definition (they literally cannot emit [PAD]).
        if pad_id is not None and 0 <= pad_id < probs.shape[-1]:
            mp += probs[pad_id].item()
        bare = [i for i in bare if i < probs.shape[-1]]
        spaced = [i for i in spaced if i < probs.shape[-1]]
        if bare:   mb += probs[bare].sum().item()
        if spaced: ms += probs[spaced].sum().item()
        n += 1
    del model; torch.cuda.empty_cache()
    return {'checkpoint': ckpt, 'n': n,
            'mass_pad': round(mp / n, 5), 'mass_bare': round(mb / n, 5), 'mass_spaced': round(ms / n, 5)}

tok = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
if tok.pad_token is None:
    tok.add_special_tokens({'pad_token': '[PAD]'})
D1_KEYS = ['base', 'A1_aug_s42', 'A2_fixed_s42', 'diff_ncp_old']   # clean vs buggy contrast
d1_rows = [d1_probe(p, SYN_CONCEPT_VAL, tok) for k, p in existing(D1_KEYS).items()]
for r, k in zip(d1_rows, existing(D1_KEYS)):
    r['arm'] = LABELS.get(k, k)
json.dump(d1_rows, open(f'{RESULTS_DIR}/d1_mass_probe.json', 'w'), indent=2)
display(pd.DataFrame(d1_rows)[['arm', 'mass_pad', 'mass_bare', 'mass_spaced', 'n']])
print('Expect: buggy arms high mass_pad/mass_bare; clean arms higher mass_spaced.')

`torch_dtype` is deprecated! Use `dtype` instead!


,arm,mass_pad,mass_bare,mass_spaced,n
0,Base Llama-3.2-1B (untrained),0.00000,0.00010,0.01690,200
1,A1 data-aug CLM (s42),0.00000,0.00006,0.04663,200
2,A2 fixed set-marginal (s42),0.00000,0.00010,0.02685,200
3,"diff_ncp (revised, buggy)",0.00002,0.06372,0.00294,200


Expect: buggy arms high mass_pad/mass_bare; clean arms higher mass_spaced.


In [ ]:
# 10b. D3 — audit sampler (CPU). Writes slots to annotate by hand; fill 'valid' with 1/0.
import pandas as pd, ast, random
def sample_audit(csv, tag, n=50, seed=42):
    df = pd.read_csv(csv); col = 'context_syn' if 'context_syn' in df.columns else df.columns[-1]
    rng = random.Random(seed); idx = rng.sample(range(len(df)), min(n, len(df)))
    rows = []
    for i in idx:
        try: words = ast.literal_eval(str(df.iloc[i][col]))
        except Exception: words = []
        rows.append({'set': tag, 'text': str(df.iloc[i]['text']),
                     'positives': ', '.join(map(str, words)), 'valid_count': '', 'notes': ''})
    return rows
audit = sample_audit(SYN_CONCEPT_VAL, 'syn') + sample_audit(HYP_CONCEPT_VAL, 'hyp')
AUDIT_CSV = f'{RESULTS_DIR}/D3_positive_audit_TOFILL.csv'
pd.DataFrame(audit).to_csv(AUDIT_CSV, index=False)
print(f'Wrote {len(audit)} slots -> {AUDIT_CSV}. Annotate valid_count (# of positives valid in context).')

Wrote 100 slots -> /content/drive/MyDrive/concept_aware_outputs/planA_results/D3_positive_audit_TOFILL.csv. Annotate valid_count (# of positives valid in context).


In [ ]:
# 10d. D3 summary — run AFTER you fill the valid_count column in the audit CSV and re-upload it.
# valid_count = how many of a slot's listed positives are genuinely valid substitutions in context.
import pandas as pd
AUDIT = f'{RESULTS_DIR}/D3_positive_audit_TOFILL.csv'   # <- point at your FILLED file
df = pd.read_csv(AUDIT)
df['valid_count'] = pd.to_numeric(df['valid_count'], errors='coerce')
df = df.dropna(subset=['valid_count'])
if len(df) == 0:
    print('No annotated rows yet — fill valid_count in the CSV, re-upload, and rerun this cell.')
else:
    print(f'Annotated slots: {len(df)}')
    print(f'Slots with >=1 valid positive: {(df["valid_count"]>=1).mean()*100:.1f}%')
    print(f'Mean valid positives per slot: {df["valid_count"].mean():.2f}')
    for tag in df['set'].unique():
        sub = df[df['set'] == tag]
        print(f'  {tag}: {(sub["valid_count"]>=1).mean()*100:.1f}% have >=1 valid (n={len(sub)})')
    print('\n>80% with >=1 valid => the concept-PPL metric is well grounded. Lower => report as a '
          'caveat and/or build a filtered eval on only audited-valid slots.')


In [9]:
# 10c. Paired per-slot stats (CPU, runs on the v2 JSONs written in Task 11). Skips gracefully if
# the eval JSON is not there yet — rerun after Task 11.
import json, os, random, math
def paired_delta(json_path, arm_a, arm_b):
    # Resolve arm keys to the checkpoint basename the eval JSON is keyed by (e.g. A1_aug_s42 ->
    # 'standard_clm', base -> 'Llama-3.2-1B'); per_row_nll rows align across models because
    # eval_concept_ppl_v2 skips rows model-independently (same tokenizer for all).
    if not os.path.exists(json_path): return None
    recs = {os.path.basename(r['checkpoint'].rstrip('/')): r for r in json.load(open(json_path)) if 'per_row_nll' in r}
    ra = recs.get(os.path.basename(ARMS[arm_a].rstrip('/')))
    rb = recs.get(os.path.basename(ARMS[arm_b].rstrip('/')))
    if not ra or not rb: return None
    a, b = ra['per_row_nll'], rb['per_row_nll']
    n = min(len(a), len(b)); d = [a[i] - b[i] for i in range(n)]     # >0 => a worse than b (per slot)
    rng = random.Random(0); means = sorted(sum(rng.choices(d, k=n)) / n for _ in range(2000))
    lo, hi = means[50], means[1949]
    win = sum(1 for x in d if x < 0) / n                             # fraction of slots a beats b
    return {'a': LABELS.get(arm_a, arm_a), 'b': LABELS.get(arm_b, arm_b), 'n': n,
            'median_dNLL': round(sorted(d)[n//2], 3), 'mean_dNLL': round(sum(d)/n, 3),
            'ci95': [round(lo, 3), round(hi, 3)], 'a_beats_b_rate': round(win, 3)}
print('paired_delta(json, arm_a, arm_b) ready — call in Task 11 after the eval JSONs exist.')

paired_delta(json, arm_a, arm_b) ready — call in Task 11 after the eval JSONs exist.


---
## Task 11 — Intrinsic dual eval (v2) + master table

Same canonical tokenizer, no resize, in-context continuation scoring, bootstrap CIs. Reports
`per_row_nll` so the paired stats in 10c can run. **Headline comparison: A2 vs A1 (paired).**

In [10]:
# 11a. Dual eval on clean syn + hyp val sets, all Plan-A arms
EVAL_KEYS = (['base']
             + [f'A0_vanilla_s{s}' for s in SEEDS]
             + ['A1_aug_s42', 'A1_aug_s123', 'A1_aug_s7']
             + [f'A2_fixed_s{s}' for s in SEEDS]
             + ['D2_alpha0'])

ck = existing(EVAL_KEYS)
for tag, csv in [('syn', SYN_CONCEPT_VAL), ('hyp', HYP_CONCEPT_VAL)]:
    out = f'{RESULTS_DIR}/dual_eval_planA_{tag}.json'
    cmd = (f"python {SCRIPTS_DIR}/eval_concept_ppl_v2.py"
           f" --checkpoints {' '.join(ck.values())}"
           f" --tokenizer_path {TOKENIZER_PATH} --concept_csv {csv}"
           f" --vanilla_val {VANILLA_VAL} --results_json {out}")
    print(f'\n=== dual eval [{tag}] ===')
    get_ipython().system(cmd)


=== dual eval [syn] ===
2026-08-07 00:32:55.612690: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-07 00:32:55.682479: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Canonical tokenizer: /content/Llama-3.2-1B (vocab 128256)

Evaluating: /content/Llama-3.2-1B
  Loading model: /content/Llama-3.2-1B
`torch_dtype` is deprecated! Use `dtype` instead!
  [1/2] NTP eval on vanilla text ...
Generating validation split: 833 examples [00:00, 128715.56 examples/s]
Map: 100% 833/833 [00:00<00:00, 26441.96 examples/s]
Map: 

In [12]:
# 11b. Headline paired comparisons on BOTH val sets (syn + hyp). hyp is where A2 wins biggest,
# so report it with paired CIs too. median_dNLL>0 and a_beats_b_rate<0.5 => "a" worse than "b".
SYN_JSON = f'{RESULTS_DIR}/dual_eval_planA_syn.json'
HYP_JSON = f'{RESULTS_DIR}/dual_eval_planA_hyp.json'
pairs = [('A2_fixed_s42', 'A1_aug_s42'), ('A2_fixed_s42', 'base'),
         ('A1_aug_s42', 'A0_vanilla_s42'), ('A2_fixed_s42', 'D2_alpha0')]

for tag, path in [('SYN', SYN_JSON), ('HYP', HYP_JSON)]:
    rows = [r for r in (paired_delta(path, a, b) for a, b in pairs) if r]
    print(f'\n=== Paired per-slot deltas on the {tag} concept set ===')
    if rows:
        display(pd.DataFrame(rows))
    else:
        print(f'  {tag}: run 11a first (no {path}).')
print('\nRead: a_beats_b_rate>0.5 and CI excluding 0 => "a" significantly better than "b" per slot.')



=== Paired per-slot deltas on the SYN concept set ===


,a,b,n,median_dNLL,mean_dNLL,ci95,a_beats_b_rate
0,A2 fixed set-marginal (s42),A1 data-aug CLM (s42),258,-1.630,-0.017,"[-1.485, 1.769]",0.729
1,A2 fixed set-marginal (s42),Base Llama-3.2-1B (untrained),258,-1.404,-1.478,"[-1.756, -1.156]",0.853
2,A1 data-aug CLM (s42),A0 vanilla CLM (s42),258,-1.750,-5.201,"[-7.893, -3.038]",0.806
3,A2 fixed set-marginal (s42),D2 forgetting control (α=0),258,-1.831,-2.154,"[-2.463, -1.852]",0.833



=== Paired per-slot deltas on the HYP concept set ===


,a,b,n,median_dNLL,mean_dNLL,ci95,a_beats_b_rate
0,A2 fixed set-marginal (s42),A1 data-aug CLM (s42),313,-2.211,-1.890,"[-2.441, -1.211]",0.796
1,A2 fixed set-marginal (s42),Base Llama-3.2-1B (untrained),313,-1.742,-2.001,"[-2.251, -1.767]",0.898
2,A1 data-aug CLM (s42),A0 vanilla CLM (s42),313,-1.755,-2.981,"[-4.127, -2.141]",0.780
3,A2 fixed set-marginal (s42),D2 forgetting control (α=0),313,-2.493,-2.593,"[-2.887, -2.319]",0.891



Read: a_beats_b_rate>0.5 and CI excluding 0 => "a" significantly better than "b" per slot.


In [11]:
# 11c. Master table (self-contained). Aggregates seeds by resolving arm keys -> checkpoint basenames
# (A1_aug_s42 -> 'standard_clm', base -> 'Llama-3.2-1B'), so the grouping is correct.
import json, os, statistics
def load(path):
    return {os.path.basename(r['checkpoint'].rstrip('/')): r
            for r in json.load(open(path))} if os.path.exists(path) else {}
syn = load(f'{RESULTS_DIR}/dual_eval_planA_syn.json')
hyp = load(f'{RESULTS_DIR}/dual_eval_planA_hyp.json')

GROUPS = [
    ('Base (untrained)',        ['base']),
    ('A0 vanilla CLM',          [f'A0_vanilla_s{s}' for s in SEEDS]),
    ('A1 data-aug CLM',         ['A1_aug_s42', 'A1_aug_s123', 'A1_aug_s7']),
    ('A2 fixed set-marginal',   [f'A2_fixed_s{s}' for s in SEEDS]),
    ('D2 α=0 control',          ['D2_alpha0']),
    ('diff_ncp (buggy, ref)',   ['diff_ncp_old']),
]
def agg(arm_keys, src, field):
    bns = [os.path.basename(ARMS[k].rstrip('/')) for k in arm_keys if k in ARMS]
    vs = [src[bn][field] for bn in bns if bn in src and field in src[bn]]
    if not vs: return '-'
    return f'{statistics.mean(vs):.2f}' + (f' ±{statistics.pstdev(vs):.2f}' if len(vs) > 1 else '')
rows = []
for label, keys in GROUPS:
    rows.append({'Arm': label, 'NTP PPL': agg(keys, syn, 'ntp_ppl'), 'NTP Acc': agg(keys, syn, 'ntp_acc'),
                 'Syn cPPL': agg(keys, syn, 'concept_ppl'), 'Syn SetMass': agg(keys, syn, 'concept_set_mass_mean'),
                 'Hyp cPPL': agg(keys, hyp, 'concept_ppl')})
df_master = pd.DataFrame(rows)
display(df_master)
df_master.to_csv(f'{OUTPUT_ROOT}/master_planA.csv', index=False)
print('saved master_planA.csv')

,Arm,NTP PPL,NTP Acc,Syn cPPL,Syn SetMass,Hyp cPPL
0,Base (untrained),7.53,0.67,18081.36,0.02,5395.39
1,A0 vanilla CLM,17.50 ±1.15,0.63 ±0.01,703078.05 ±45502.76,0.02 ±0.00,86599.65 ±8066.04
2,A1 data-aug CLM,17.54 ±0.84,0.62 ±0.00,4285.29 ±76.99,0.04 ±0.00,4967.90 ±202.31
3,A2 fixed set-marginal,8.28 ±0.67,0.67 ±0.00,15028.10 ±15352.11,0.02 ±0.01,3752.83 ±4266.73
4,D2 α=0 control,7.30,0.67,35556.95,0.02,9757.09
5,"diff_ncp (buggy, ref)",-,-,-,-,-


saved master_planA.csv


---
## Task 12 — Downstream + representation / retention (Chen's suggestions)

- **SNLI** linear probe + low-resource FT (the July signal: concept models beat CLM at n=500).
- **R1** synonym-invariance gap on our own data (representation-level concept test).
- **R2** STS-B (external representation validity; relative ordering only).
- **R3** ARC/MMLU/etc as **capability RETENTION** (gains not expected at 1B×2K rows; damage detection).

All downstream/probe checkpoints go to LOCAL scratch — only the result JSONs reach Drive.

In [ ]:
# 12a. SNLI linear probe (primary transfer metric). Probes written to local scratch, JSON to Drive.
DS_KEYS = ['base', 'A0_vanilla_s42', 'A1_aug_s42', 'A2_fixed_s42']
ck = existing(DS_KEYS)
out = f'{RESULTS_DIR}/snli_probe.json'
cmd = (f"python {SCRIPTS_DIR}/run_downstream_eval.py --checkpoints {' '.join(ck.values())}"
       f" --task snli --freeze_base --max_train_samples 20000 --num_epochs 5"
       f" --output_dir {LOCAL_SCRATCH}/snli_probe --results_json {out}")
get_ipython().system(cmd)
print(open(out).read()[:400] if os.path.exists(out) else 'no json')


Checkpoint: /content/Llama-3.2-1B
  Loading tokenizer + model: /content/Llama-3.2-1B
`torch_dtype` is deprecated! Use `dtype` instead!
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
  Loading snli dataset ...
README.md: 16.0kB [00:00, 40.5MB/s]
plain_text/test-00000-of-00001.parquet: 100% 412k/412k [00:01<00:00, 309kB/s]
plain_text/validation-00000-of-00001.par(…): 100% 413k/413k [00:00<00:00, 897kB/s]
plain_text/train-00000-of-00001.parquet: 100% 19.6M/19.6M [00:00<00:00, 24.7MB/s]
Generating test split: 100% 10000/10000 [00:00<00:00, 205250.99 examples/s]
Generating validation split: 100% 10000/10000 [00:00<00:00, 1676581.52 examples/s]
Generating train split: 100% 550152/550152 [00:00<00:00, 2848373.91 examples/s]
Filter: 100% 10000/10000 [00:00<00:00, 241782.86 examples/s]
Filt

In [ ]:
# 12b. SNLI low-resource fine-tune (n=500 across arms; add 100/1000 for seed42 if budget allows)
for n in [500]:
    out = f'{RESULTS_DIR}/snli_lowres_{n}.json'
    cmd = (f"python {SCRIPTS_DIR}/run_downstream_eval.py --checkpoints {' '.join(ck.values())}"
           f" --task snli --max_train_samples {n} --num_epochs 10"
           f" --output_dir {LOCAL_SCRATCH}/snli_lowres --results_json {out}")
    print(f'\n=== SNLI FT n={n} ==='); get_ipython().system(cmd)


=== SNLI FT n=500 ===

Checkpoint: /content/Llama-3.2-1B
  Loading tokenizer + model: /content/Llama-3.2-1B
`torch_dtype` is deprecated! Use `dtype` instead!
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
  Loading snli dataset ...
Map: 100% 9842/9842 [00:00<00:00, 12501.30 examples/s]
Map: 100% 500/500 [00:00<00:00, 11231.59 examples/s]
  Training (full fine-tune) ...
{'eval_loss': 0.8133507370948792, 'eval_accuracy': 0.6484454379191221, 'eval_f1': 0.648520133789241, 'eval_runtime': 86.6872, 'eval_samples_per_second': 113.535, 'eval_steps_per_second': 3.553, 'epoch': 1.0}
{'eval_loss': 0.8929030895233154, 'eval_accuracy': 0.6568786831944726, 'eval_f1': 0.6488979807916014, 'eval_runtime': 86.532, 'eval_samples_per_second': 113.738, 'eval_steps_per_second': 3.559, 'epoch': 2.0}
{'ev

In [ ]:
# 12c. R1 — synonym-invariance gap (FIXED probe). Two changes vs the first version:
# (1) use the hidden state of the CONCEPT-WORD token (last real token), not a mean-pool of the whole
#     sentence — the old mean-pool was ~12/13 shared context, so it barely saw the concept word;
# (2) CENTER embeddings on their global mean before cosine, to remove decoder anisotropy (raw cosines
#     were all ~0.99 = uninformative). Read arms RELATIVELY; gap_winrate is the robust readout.
import torch, ast, json
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from scipy.stats import spearmanr

@torch.no_grad()
def embed_lasttok(model, tok, sents, bs=16):
    out = []
    for i in range(0, len(sents), bs):
        b = tok(sents[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=64).to('cuda')
        h = model(**b, output_hidden_states=True).hidden_states[-1]        # [B,L,H]
        last = b['attention_mask'].sum(1) - 1                              # index of last real token
        pooled = h[torch.arange(h.size(0), device=h.device), last]        # [B,H] concept-word rep
        out.append(pooled.float().cpu())
    return torch.cat(out)

@torch.no_grad()
def r1_gap(ckpt, tok, n=200):
    model = AutoModelForCausalLM.from_pretrained(ckpt, torch_dtype=torch.bfloat16).cuda().eval()
    df = pd.read_csv(CONTRA_MERGED_VAL)
    gold, syn, neg = [], [], []
    for _, row in df.head(n).iterrows():
        try:
            pos = ast.literal_eval(str(row['positives'])); negs = ast.literal_eval(str(row['negatives']))
        except Exception: continue
        if len(pos) < 2 or not negs: continue
        ctx = str(row['text']).rstrip()
        gold.append(f'{ctx} {pos[0]}'); syn.append(f'{ctx} {pos[1]}'); neg.append(f'{ctx} {negs[0]}')
    if not gold:
        del model; torch.cuda.empty_cache(); return None
    eg, es, en = embed_lasttok(model, tok, gold), embed_lasttok(model, tok, syn), embed_lasttok(model, tok, neg)
    del model; torch.cuda.empty_cache()
    mu = torch.cat([eg, es, en]).mean(0, keepdim=True)                     # center (kill anisotropy)
    eg, es, en = F.normalize(eg-mu, dim=-1), F.normalize(es-mu, dim=-1), F.normalize(en-mu, dim=-1)
    cs = (eg*es).sum(-1); cn = (eg*en).sum(-1)
    return {'checkpoint': ckpt, 'n': len(gold), 'cos_syn': round(cs.mean().item(),4),
            'cos_neg': round(cn.mean().item(),4), 'gap': round((cs-cn).mean().item(),4),
            'gap_winrate': round((cs>cn).float().mean().item(),3)}

# Build merged contrastive val if the revised-round Phase 1 didn't leave it in the repo.
if not os.path.exists(CONTRA_MERGED_VAL):
    print('contrastive/youtube_clean missing — building it from the clean splits...')
    get_ipython().system(
        f"python {REPO_DIR}/build_contrastive_dataset.py"
        f" --syn_train {SYN_CONCEPT_TRAIN} --syn_val {SYN_CONCEPT_VAL}"
        f" --hyp_train {CLEAN_HYP}/context_loss_train.csv --hyp_val {HYP_CONCEPT_VAL}"
        f" --source both --strategy all --max_negatives 10"
        f" --output_dir {DATA_ROOT}/contrastive/youtube_clean")
assert os.path.exists(CONTRA_MERGED_VAL), f'still missing: {CONTRA_MERGED_VAL}'

tok = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
tok.pad_token = tok.eos_token
tok.padding_side = 'right'
R1_KEYS = ['base', 'A0_vanilla_s42', 'A1_aug_s42', 'A2_fixed_s42', 'diff_ncp_old']
r1 = [x for x in (r1_gap(p, tok) for p in existing(R1_KEYS).values()) if x]
for r, k in zip(r1, existing(R1_KEYS)): r['arm'] = LABELS.get(k, k)
json.dump(r1, open(f'{RESULTS_DIR}/r1_synonym_gap.json','w'), indent=2)
display(pd.DataFrame(r1)[['arm','cos_syn','cos_neg','gap','gap_winrate','n']])
print('gap>0 and gap_winrate>0.5 => reps place synonyms closer than negatives. Judge arms RELATIVELY, '
      'not by absolute value. If gap_winrate ~0.5 for everyone, reps are uninformative -> drop R1.')


In [ ]:
# 12d. R2 — STS-B (GPU, eval-only). Spearman of cos(sent1,sent2) vs gold. Relative ordering matters.
from datasets import load_dataset
sts = load_dataset('glue', 'stsb', split='validation')
s1, s2, y = list(sts['sentence1']), list(sts['sentence2']), list(sts['label'])
def r2_sts(ckpt, tok):
    model = AutoModelForCausalLM.from_pretrained(ckpt, torch_dtype=torch.bfloat16).cuda().eval()
    e1, e2 = embed(model, tok, s1), embed(model, tok, s2)
    cos = (e1 * e2).sum(-1).numpy()
    del model; torch.cuda.empty_cache()
    return round(spearmanr(cos, y).correlation, 4)
r2 = []
for k, p in existing(R1_KEYS).items():
    r2.append({'arm': LABELS.get(k, k), 'stsb_spearman': r2_sts(p, tok)})
json.dump(r2, open(f'{RESULTS_DIR}/r2_stsb.json', 'w'), indent=2)
display(pd.DataFrame(r2))

README.md: 0.00B [00:00, ?B/s]

stsb/train-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

stsb/validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

stsb/test-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

,arm,stsb_spearman
0,Base Llama-3.2-1B (untrained),0.5800
1,A0 vanilla CLM (s42),0.6839
2,A1 data-aug CLM (s42),0.6630
3,A2 fixed set-marginal (s42),0.5717
4,"diff_ncp (revised, buggy)",0.6679


In [ ]:
# 12e. R3 — capability RETENTION via lm-eval-harness (GPU, SLOW). Gains not expected at 1B×2K rows;
# this detects damage. Run on a small model set to save budget. Needs: pip install lm-eval (S2).
# MMLU is slow + near-floor for 1B — left commented.
R3_KEYS = ['base', 'A1_aug_s42', 'A2_fixed_s42']
TASKS = 'arc_easy,arc_challenge,hellaswag,piqa,winogrande'   # add ',mmlu' only on A100 / with time
for k, p in existing(R3_KEYS).items():
    out = f'{RESULTS_DIR}/r3_{k}'
    cmd = (f"lm_eval --model hf --model_args pretrained={p},dtype=bfloat16"
           f" --tasks {TASKS} --batch_size 16 --output_path {out}")
    print(f'\n=== R3 retention: {k} ==='); get_ipython().system(cmd)
print('Parse the per-task acc from the JSONs under', RESULTS_DIR)


=== R3 retention: base ===
/bin/bash: line 1: lm_eval: command not found

=== R3 retention: A1_aug_s42 ===
/bin/bash: line 1: lm_eval: command not found

=== R3 retention: A2_fixed_s42 ===
/bin/bash: line 1: lm_eval: command not found
Parse the per-task acc from the JSONs under /content/drive/MyDrive/concept_aware_outputs/planA_results


In [ ]:
# 12f. Downstream + representation summary (self-contained)
import json, os
def load_list(path):
    return json.load(open(path)) if os.path.exists(path) else []
def acc_map(path):
    return {os.path.basename(r['checkpoint'].rstrip('/')): r.get('accuracy') for r in load_list(path) if 'error' not in r}
probe = acc_map(f'{RESULTS_DIR}/snli_probe.json'); low = acc_map(f'{RESULTS_DIR}/snli_lowres_500.json')
r1 = {r['checkpoint'].split('/')[-1]: r['gap'] for r in load_list(f'{RESULTS_DIR}/r1_synonym_gap.json')}
r2 = {r['arm']: r['stsb_spearman'] for r in load_list(f'{RESULTS_DIR}/r2_stsb.json')}
rows = []
for k in ['base', 'A0_vanilla_s42', 'A1_aug_s42', 'A2_fixed_s42', 'diff_ncp_old']:
    base = os.path.basename(ARMS[k].rstrip('/'))
    rows.append({'Arm': LABELS.get(k, k),
                 'SNLI probe': probe.get(base, '-'), 'SNLI FT@500': low.get(base, '-'),
                 'R1 syn-gap': r1.get(base, '-'), 'R2 STS-B': r2.get(LABELS.get(k, k), '-')})
display(pd.DataFrame(rows))

,Arm,SNLI probe,SNLI FT@500,R1 syn-gap,R2 STS-B
0,Base Llama-3.2-1B (untrained),0.6374,0.7669,0.0057,0.5800
1,A0 vanilla CLM (s42),0.6158,0.6783,0.0167,0.6839
2,A1 data-aug CLM (s42),0.6294,0.6777,0.0161,0.6630
3,A2 fixed set-marginal (s42),0.6225,0.7562,0.0035,0.5717
4,"diff_ncp (revised, buggy)",-,-,0.0069,0.6679


---
## Compute budget & GPU strategy

**Rule of thumb at 1B params, ~2K rows:** every part here fits in T4 (16 GB) VRAM; **A100 is never
required**. L4 is the sweet spot — cheapest *per unit work* for the compute-bound cells and fast enough.
Run the CPU-only cells (9a, 10b, 10c, 11b/c, 12f) on a **CPU runtime** = 0 GPU credits.

| Section | Runs | ~min each (L4) | L4 hours | Best GPU |
|---|---|---|---|---|
| Task 9 CORE training (A0, A2, D2 — seed 42; A1 reused) | 3 | ~22 | ~1.1 | **L4** |
| Task 11 intrinsic eval (syn+hyp) + D1 | 2–3 | ~15 | ~0.8 | L4 or T4 |
| SNLI probe (4 arms) | 1×4 | ~22 | ~1.5 | L4 |
| SNLI low-res FT@500 (4 arms) | 1×4 | ~12 | ~0.8 | L4 or T4 |
| R1 + R2 (5 arms) | eval | ~6 | ~1.0 | T4 fine |
| R3 retention (3 arms, no MMLU) | 3 | ~45 | ~2.3 | **L4** |
| CPU cells (9a,10b,10c,11b,11c,12f) | — | — | 0 (CPU rt) | CPU |
| **Total (core, seed 42, no MMLU)** | | | **~7.5 L4-hr ≈ ~12 credits** | |
| *Deferred:* extra seeds (cell 9c, A0×2+A1×2+A2×2) | 6 | ~22 | ~2.2 | L4 (camera-ready) |

**Credit cost (your rates):**
- All-L4 core (seed 42) ≈ 7.5 × 1.54 ≈ **~12 credits**; extra seeds add ~2.2 hr ≈ ~3.4 credits later.
- All-T4 core ≈ ~11 hr × 1.15 ≈ **~13 credits** (T4 ~1.5× slower, so *more* credits despite lower rate
  — L4 wins for compute-bound work).
- **A100 (6.77/hr) / G4 (8.3/hr): do not use** — no OOM here; you'd pay 4–5× for no benefit. Only
  consider A100 if you add MMLU to R3 (adds ~1–2 hr/model) and want it done fast.

**Cheapest sensible split:** L4 for Tasks 9 + R3 (compute-bound); T4 for the eval-only cells if L4 is
scarce; CPU runtime for the pure-pandas cells. Expect **~12 credits** for the full core notebook (seed 42).

**Trim levers if over budget:** run SNLI probe for seed-42 arms only (already the case here); skip R3
or drop to `arc_easy,hellaswag`; defer the 3-seed expansion of A0/A1/A2 to camera-ready (seed-42 gives
the direction, extra seeds give the error bars).

In [ ]:
# Cleanup: guarantee no training-state files linger on Drive, and report footprint.
!find {OUTPUT_ROOT}/planA \( -name "optimizer.pt" -o -name "scheduler.pt" -o -name "rng_state.pth" -o -name "scaler.pt" \) -delete 2>/dev/null
!find {OUTPUT_ROOT}/planA -type d -name "checkpoint-*" -exec rm -rf {} + 2>/dev/null
print('Drive footprint (planA + results):')
!du -sh {OUTPUT_ROOT}/planA {RESULTS_DIR} 2>/dev/null
print('\nLocal scratch (ephemeral, holds the state files — not on Drive):')
!du -sh {LOCAL_SCRATCH} 2>/dev/null

---

## Chen follow-up (Aug 2026) — MTEB STS suite, ARC/HellaSwag/PIQA/Winogrande retention, and A2 stabilization

Chen's note: *harder benchmarks (ARC, MMLU); STS tasks from MTEB — try 3–4 datasets and see where we get gains.*
These cells run **on top of** everything above (they reuse `ARMS`, `LABELS`, `existing()`, `train_arm()`,
`TOKENIZER_PATH`, `RESULTS_DIR`). Nothing above is modified. Three additions:

- **R2b** — STS through the `mteb` package proper (4 datasets, canonical test splits, leaderboard-comparable)
  — supersedes the quick GLUE-sourced STS-B pass in cell 12d.
- **R3b** — capability-retention suite via `lm-eval-harness` (self-contained: installs, runs, parses).
- **Stabilization** — retrain A2 with a stronger concept weight (α=1.0) + a short warmup across 3 seeds
  to measure the failure rate of the non-engagement bug.

Compute/wall-clock per variant is in the last markdown cell of this section.

In [ ]:
# S-followup: deps for this section (mteb for R2b, lm-eval for R3b). Heavy — run once per session.
!pip install -q mteb lm-eval
print('mteb + lm-eval ready')

In [ ]:
# 12g. R2b — STS via the MTEB package proper (Chen's ask: "STS tasks from MTEB").
# Canonical MTEB test splits -> numbers are directly comparable to the MTEB leaderboard, unlike the
# quick GLUE-sourced STS-B pass in 12d. Sentence embedding = mean-pooled last hidden state (the standard
# MTEB decoder baseline); cosine + Spearman is computed inside mteb. Eval-only, GPU.
import mteb, torch, numpy as np, json
from transformers import AutoModelForCausalLM, AutoTokenizer

STS_TASKS = ['STSBenchmark', 'STS12', 'STS13', 'SICK-R']   # 4 English STS datasets (add STS14-17/BIOSSES for more)
R2B_KEYS  = ['base', 'A0_vanilla_s42', 'A1_aug_s42', 'A2_fixed_s42', 'D2_alpha0']

class LMEncoder:
    """Wrap a causal-LM checkpoint as an mteb encoder (.encode -> np array of sentence embeddings)."""
    def __init__(self, ckpt, bs=32, maxlen=64):
        self.tok = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
        self.tok.pad_token = self.tok.eos_token; self.tok.padding_side = 'right'
        self.model = AutoModelForCausalLM.from_pretrained(ckpt, torch_dtype=torch.bfloat16).cuda().eval()
        self.bs, self.maxlen = bs, maxlen
    @torch.no_grad()
    def encode(self, sentences, **kw):                       # kw absorbs task_name/prompt_type across mteb versions
        embs = []
        for i in range(0, len(sentences), self.bs):
            b = self.tok(list(sentences[i:i+self.bs]), return_tensors='pt', padding=True,
                         truncation=True, max_length=self.maxlen).to('cuda')
            h = self.model(**b, output_hidden_states=True).hidden_states[-1]      # [B,L,H]
            m = b['attention_mask'].unsqueeze(-1).float()
            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)                       # mean-pool real tokens only
            embs.append(pooled.float().cpu().numpy())
        return np.concatenate(embs)

def _sts_score(r):
    # STS main_score == cosine spearman; parse robustly across mteb versions
    try: return float(r.get_score())
    except Exception: pass
    sc = getattr(r, 'scores', r)
    try:
        d = sc['test'][0]
        return float(d.get('main_score') or d.get('cosine_spearman') or d.get('cos_sim', {}).get('spearman'))
    except Exception:
        return float('nan')

tasks, rows = mteb.get_tasks(tasks=STS_TASKS), []
for k, p in existing(R2B_KEYS).items():
    enc = LMEncoder(p)
    res = mteb.MTEB(tasks=tasks).run(enc, output_folder=f'{RESULTS_DIR}/mteb/{k}', eval_splits=['test'])
    row = {'arm': LABELS.get(k, k)}
    for r in res: row[r.task_name] = round(_sts_score(r), 4)
    rows.append(row)
    del enc; torch.cuda.empty_cache()
r2b = pd.DataFrame(rows); json.dump(rows, open(f'{RESULTS_DIR}/r2b_mteb_sts.json', 'w'), indent=2)
display(r2b)
print('cosine-Spearman per STS task (MTEB test splits). base vs A2_fixed = the concept-training effect. '
      'Judge arms RELATIVELY; look for tasks where A2 > A1/A0 ("where we get good gains").')

In [ ]:
# 12h. R3b — capability RETENTION via lm-eval-harness (self-contained: install -> run -> parse).
# ARC-easy/ARC-challenge/HellaSwag/PIQA/Winogrande. Gains are NOT the hypothesis at 1B x 2.2K rows —
# this detects DAMAGE (how much base capability each objective preserves). MMLU left off (near-floor
# at 1B, ~1-2h/model extra) — enable on A100 for camera-ready.
import json, glob, os
try:
    import lm_eval  # noqa: F401
except ImportError:
    get_ipython().system('pip install -q lm-eval')

R3_KEYS = ['base', 'A1_aug_s42', 'A2_fixed_s42']            # small set to save budget; add A0/seeds if time
TASKS   = 'arc_easy,arc_challenge,hellaswag,piqa,winogrande'
for k, p in existing(R3_KEYS).items():
    out = f'{RESULTS_DIR}/r3/{k}'; os.makedirs(out, exist_ok=True)
    cmd = (f"lm_eval --model hf --model_args pretrained={p},dtype=bfloat16"
           f" --tasks {TASKS} --batch_size 8 --output_path {out}")
    print(f'\n=== R3 retention: {k} ==='); get_ipython().system(cmd)

# parse the per-task accuracy from lm-eval's results JSON(s)
METRIC = {'arc_easy': 'acc', 'arc_challenge': 'acc_norm', 'hellaswag': 'acc_norm', 'piqa': 'acc', 'winogrande': 'acc'}
rows = []
for k in existing(R3_KEYS):
    js = sorted(glob.glob(f'{RESULTS_DIR}/r3/{k}/**/results*.json', recursive=True))
    if not js: continue
    res = json.load(open(js[-1])).get('results', {})
    row = {'arm': LABELS.get(k, k)}
    for t, m in METRIC.items():
        d = res.get(t, {})
        row[t] = round(d.get(f'{m},none', d.get(m, float('nan'))), 4)
    rows.append(row)
if rows:
    r3 = pd.DataFrame(rows); json.dump(rows, open(f'{RESULTS_DIR}/r3_retention.json', 'w'), indent=2)
    display(r3)
    print('Accuracy per task. Read as RETENTION: A1/A2 close to base = capability preserved; a large '
          'drop = the objective damaged general ability. Not expecting gains at this scale.')
else:
    print('No results JSONs found — check the lm_eval runs above completed.')

### Stabilization — fixing the A2 non-engagement seed

Revised-round finding (the email): under the gentle schedule (α = 0.5, 2 epochs, lr 1e-5) that protects
NTP quality, **one seed in three failed to engage the concept term** and collapsed to the α = 0 control
(hyp concept PPL ~9,790 instead of ~730). Hypothesis: optimization, not fundamental. Fix = a **stronger
concept weight (α = 1.0) + a short warmup**, run across seeds to measure the failure rate directly.

In [ ]:
# 12i. STABILIZATION training — A2 with α=1.0 + 10% warmup, across 3 seeds (registers new arms A2b_*).
STAB_SEEDS = [42, 123, 7]
STAB_EXTRA = '--ncp_alpha 1.0 --warmup_ratio 0.1'
for s in STAB_SEEDS:
    key = f'A2b_strong_s{s}'
    ARMS[key]   = f'{PLANA_OUT}/{key}'                      # register so train_arm/existing() see it
    LABELS[key] = f'A2b strong-α+warmup (s{s})'
    train_arm(key, 'run_clm_differentiable_ncp.py', extra_args=STAB_EXTRA,
              seed=s, train_file=A2_TRAIN, val_file=A2_VAL, epochs=2, lr=1e-5)
print('\nStabilized arms trained:', [f'A2b_strong_s{s}' for s in STAB_SEEDS])

In [ ]:
# 12j. Did the fix engage? Concept-PPL (v2) on the stabilized arms vs original A2 + the α=0 control.
# Failure signature = a seed whose syn/hyp concept PPL sits up at the D2 (α=0) level.
STAB_KEYS = [f'A2b_strong_s{s}' for s in STAB_SEEDS] + ['A2_fixed_s42', 'D2_alpha0', 'A1_aug_s42']
ck = existing(STAB_KEYS)
for tag, csv in [('syn', SYN_CONCEPT_VAL), ('hyp', HYP_CONCEPT_VAL)]:
    out = f'{RESULTS_DIR}/stab_eval_{tag}.json'
    cmd = (f"python {SCRIPTS_DIR}/eval_concept_ppl_v2.py"
           f" --checkpoints {' '.join(ck.values())}"
           f" --tokenizer_path {TOKENIZER_PATH} --concept_csv {csv}"
           f" --vanilla_val {VANILLA_VAL} --results_json {out}")
    print(f'\n=== stabilization eval [{tag}] ==='); get_ipython().system(cmd)
print('Compare each A2b seed to D2 (α=0) on hyp concept PPL. All three well below D2 => warmup+stronger-α '
      'fixed non-engagement; count any that still collapse = residual failure rate.')

### Stabilization ablation — which lever, and how reliable?

`12i` combines a stronger concept weight **and** a warmup in one arm, so it can confirm the fix works but
can't say which lever did it. These two cells run the levers **separately** across seeds — implementing the
plan's *"a stronger concept weight **or** a short warmup, then more seeds to characterize the failure rate."*

In [ ]:
# 12k. STABILIZATION ABLATION — isolate WHICH lever fixes the non-engagement seed, separately, and
# sweep seeds to characterize the failure rate. Baseline = A2_fixed (α=0.5, no warmup) = the config
# that failed on 1 of 3 seeds. Registers new arms (Sa_/Sw_/Sb_*).
#   Sa_alpha : α=1.0, no warmup     (stronger concept weight only)
#   Sw_warm  : α=0.5, +10% warmup   (short warmup only)
#   Sb_both  : α=1.0, +10% warmup   (both — same recipe as 12i)
STAB_GRID  = {'Sa_alpha': '--ncp_alpha 1.0',
              'Sw_warm':  '--ncp_alpha 0.5 --warmup_ratio 0.1',
              'Sb_both':  '--ncp_alpha 1.0 --warmup_ratio 0.1'}
STAB_SEEDS = [42, 123, 7]     # add more (e.g. + [1, 2, 5]) for a tighter failure-rate estimate — see compute note
for cfg, extra in STAB_GRID.items():
    for s in STAB_SEEDS:
        key = f'{cfg}_s{s}'
        ARMS[key]   = f'{PLANA_OUT}/{key}'                 # register so train_arm/existing() see it
        LABELS[key] = f'{cfg} (s{s})'
        train_arm(key, 'run_clm_differentiable_ncp.py', extra_args=extra,
                  seed=s, train_file=A2_TRAIN, val_file=A2_VAL, epochs=2, lr=1e-5)
print('\nAblation grid trained:', [f'{c}_s{s}' for c in STAB_GRID for s in STAB_SEEDS])

In [ ]:
# 12l. FAILURE-RATE readout. Concept-PPL (v2) on every grid arm + the α=0 control (D2) + A1 reference,
# on the HYP set (where the non-engagement showed). A seed "failed to engage" if its hyp concept PPL is
# within 20% of the α=0 control. Reports engaged/total per config -> which lever is most reliable.
import json
GRID_KEYS = [f'{c}_s{s}' for c in STAB_GRID for s in STAB_SEEDS] + ['A2_fixed_s42', 'D2_alpha0', 'A1_aug_s42']
ck  = existing(GRID_KEYS)
out = f'{RESULTS_DIR}/stab_ablation_hyp.json'
get_ipython().system(
    f"python {SCRIPTS_DIR}/eval_concept_ppl_v2.py --checkpoints {' '.join(ck.values())}"
    f" --tokenizer_path {TOKENIZER_PATH} --concept_csv {HYP_CONCEPT_VAL}"
    f" --vanilla_val {VANILLA_VAL} --results_json {out}")

res = json.load(open(out))
ppl = {os.path.basename(r['checkpoint'].rstrip('/')): r['concept_ppl'] for r in res if 'concept_ppl' in r}
d2  = ppl.get('D2_alpha0', float('inf'))                    # α=0 reference = "did not engage"
thresh = 0.8 * d2                                           # engaged := hyp concept PPL < 80% of α=0
print(f'\nα=0 control (D2) hyp concept PPL = {d2:.0f}; count as engaged if < {thresh:.0f}\n')
rows = []
for cfg in STAB_GRID:
    seeds   = {s: ppl[f'{cfg}_s{s}'] for s in STAB_SEEDS if f'{cfg}_s{s}' in ppl}
    engaged = [s for s, v in seeds.items() if v < thresh]
    rows.append({'config': cfg, 'engaged': f'{len(engaged)}/{len(seeds)}',
                 'ppl_by_seed': {s: round(v) for s, v in seeds.items()}})
display(pd.DataFrame(rows))
print('Highest engaged fraction = most reliable lever. A2_fixed baseline (α=0.5, no warmup) = '
      f"{ppl.get('A2_fixed_s42', float('nan')):.0f}. Extend STAB_SEEDS for a tighter failure-rate estimate.")

### Compute & wall-clock per dataset variant

Estimates for **Colab T4 (16 GB)** on Llama-3.2-1B, bf16. An **A100** is roughly **3–4× faster**.
Eval cells are inference-only (light VRAM); training cells need the usual ~5 GB.

| Variant | What runs | Per dataset × model | Full cell (as written) |
|---|---|---|---|
| **R2b — MTEB STS** (`12g`) | embed + cosine/Spearman, eval-only | ~2–4 min | 4 tasks × 5 arms ≈ **1–1.5 h** |
| &nbsp;&nbsp;STSBenchmark | 1,379 test pairs | ~2 min/arm | — |
| &nbsp;&nbsp;STS12 / STS13 | ~3,100 / ~1,500 pairs | ~2–3 / ~2 min/arm | — |
| &nbsp;&nbsp;SICK-R | ~4,900 test pairs | ~3–4 min/arm | — |
| **R3b — retention** (`12h`) | lm-eval-harness, eval-only | see below | 3 arms ≈ **2–3.5 h** |
| &nbsp;&nbsp;arc_easy / arc_challenge | ~2,376 / ~1,172 | ~6 / ~5 min/model | — |
| &nbsp;&nbsp;hellaswag | ~10,000 (4-way) | **~25–40 min/model** (dominates) | — |
| &nbsp;&nbsp;piqa / winogrande | ~1,838 / ~1,267 | ~6 / ~5 min/model | — |
| &nbsp;&nbsp;*MMLU (deferred)* | ~14,000 (4-way) | *~1–2 h/model* | *A100 / camera-ready only* |
| **Stabilization train** (`12i`) | 2 ep × ~2.5 K rows, α=1.0 | ~25–40 min/seed | 3 seeds ≈ **1.5–2 h** |
| **Stabilization eval** (`12j`) | concept-PPL v2, syn+hyp | ~2–4 min/ckpt | 6 ckpts × 2 sets ≈ **0.5–0.75 h** |

**Session budget:** R2b + R3b together ≈ **3–5 h** on one T4 (eval-only, can run unattended). The
stabilization block adds ≈ **2–2.75 h** of training+eval. If splitting across sessions: run R2b + the
stabilization block first (they answer Chen's representation question + the variance caveat), then R3b
retention. Enable MMLU only on an A100.